In [23]:
from pathlib import Path
import numpy as np

BASE_DIR = Path("/mnt/c/Project/school/bigdata/BigData_Nhom_6")

EMB_PATH = BASE_DIR / "scripts/benchmark/data/faiss/full_embeddings.npy"

FLAT_PATH = "/mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/flat_full.index"
HNSW_PATH = "/mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/hnsw_full.index"
x_all = np.load(EMB_PATH, mmap_mode="r")

print("embedding shape:", x_all.shape)
print("embedding dtype:", x_all.dtype)

embedding shape: (1347729, 384)
embedding dtype: float32


# Build lại FlatIP full

In [ ]:
import faiss
import time
import gc

x_all = np.load(EMB_PATH, mmap_mode="r")
n_total, d = x_all.shape

assert d == 384

BATCH_SIZE = 20_000

flat = faiss.IndexFlatIP(d)

t0 = time.perf_counter()

for start in range(0, n_total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_total)

    batch = np.array(x_all[start:end], dtype="float32", copy=True)
    batch = np.ascontiguousarray(batch)
    faiss.normalize_L2(batch)

    flat.add(batch)

    if end % 200_000 == 0 or end == n_total:
        print(f"Flat added {end:,}/{n_total:,}")

build_sec = time.perf_counter() - t0

print("flat dim:", flat.d)
print("flat ntotal:", flat.ntotal)
print("build_sec:", round(build_sec, 2))

assert flat.d == 384
assert flat.ntotal == n_total

faiss.write_index(flat, str(FLAT_PATH))

print("saved:", FLAT_PATH)

In [24]:
test_flat = faiss.read_index(str(FLAT_PATH))

print("loaded flat dim:", test_flat.d)
print("loaded flat ntotal:", test_flat.ntotal)

assert test_flat.d == 384
assert test_flat.ntotal == n_total

del test_flat
gc.collect()

loaded flat dim: 384
loaded flat ntotal: 1347729


11

## Build lại HNSW full

In [ ]:
M = 32
EF_CONSTRUCTION = 80
EF_SEARCH = 64
BATCH_SIZE = 100_000

hnsw = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = EF_CONSTRUCTION
hnsw.hnsw.efSearch = EF_SEARCH

t0 = time.perf_counter()

for start in range(0, n_total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_total)

    batch = np.array(x_all[start:end], dtype="float32", copy=True)
    batch = np.ascontiguousarray(batch)
    faiss.normalize_L2(batch)

    hnsw.add(batch)

    elapsed = time.perf_counter() - t0
    speed = end / elapsed
    remain = (n_total - end) / speed

    print(
        f"HNSW added {end:,}/{n_total:,} | "
        f"elapsed {elapsed/60:.1f} min | "
        f"eta {remain/60:.1f} min"
    )

build_sec = time.perf_counter() - t0

print("hnsw dim:", hnsw.d)
print("hnsw ntotal:", hnsw.ntotal)
print("build_sec:", round(build_sec, 2))

assert hnsw.d == 384
assert hnsw.ntotal == n_total


faiss.write_index(hnsw, str(HNSW_PATH))

print("saved:", HNSW_PATH)


HNSW added 100,000/1,347,729 | elapsed 0.4 min | eta 4.6 min
HNSW added 200,000/1,347,729 | elapsed 0.8 min | eta 4.7 min
HNSW added 300,000/1,347,729 | elapsed 1.3 min | eta 4.5 min
HNSW added 400,000/1,347,729 | elapsed 1.7 min | eta 4.1 min
HNSW added 500,000/1,347,729 | elapsed 2.2 min | eta 3.7 min
HNSW added 600,000/1,347,729 | elapsed 2.7 min | eta 3.3 min
HNSW added 700,000/1,347,729 | elapsed 3.1 min | eta 2.9 min
HNSW added 800,000/1,347,729 | elapsed 3.6 min | eta 2.5 min
HNSW added 900,000/1,347,729 | elapsed 4.2 min | eta 2.1 min
HNSW added 1,000,000/1,347,729 | elapsed 4.6 min | eta 1.6 min
HNSW added 1,100,000/1,347,729 | elapsed 5.1 min | eta 1.1 min
HNSW added 1,200,000/1,347,729 | elapsed 5.6 min | eta 0.7 min
HNSW added 1,300,000/1,347,729 | elapsed 6.1 min | eta 0.2 min
HNSW added 1,347,729/1,347,729 | elapsed 6.3 min | eta 0.0 min
hnsw dim: 384
hnsw ntotal: 1347729
build_sec: 376.84
saved: /mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendatio

In [25]:
test_hnsw = faiss.read_index(str(HNSW_PATH))

print("loaded hnsw dim:", test_hnsw.d)
print("loaded hnsw ntotal:", test_hnsw.ntotal)

assert test_hnsw.d == 384
assert test_hnsw.ntotal == n_total

del test_hnsw
gc.collect()

loaded hnsw dim: 384
loaded hnsw ntotal: 1347729


0

## Cấu hình so sánh search

In [30]:
for p in [FLAT_PATH, HNSW_PATH]:
    idx = faiss.read_index(str(p))
    print(p)
    print("dim:", idx.d)
    print("ntotal:", idx.ntotal)
    print()
    del idx
    gc.collect()

/mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/flat_full.index
dim: 384
ntotal: 1347729

/mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/hnsw_full.index
dim: 384
ntotal: 1347729



## Tạo query sample

In [ ]:
TOPK = 10
N_QUERIES = 500
SEED = 42

EF_SEARCH = 64
IVF_NPROBE = 64


COMPARE_OUT = "/mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/outputs/benchmark/compare_flat_vs_hnsw_full.csv"

In [27]:
x_all = np.load(EMB_PATH, mmap_mode="r")

rng = np.random.default_rng(SEED)
qids = rng.choice(x_all.shape[0], size=N_QUERIES, replace=False)

xq = np.array(x_all[qids], dtype="float32", copy=True)
xq = np.ascontiguousarray(xq)

faiss.normalize_L2(xq)

print("xq shape:", xq.shape)

xq shape: (500, 384)


In [32]:
def set_search_params(index):
    if hasattr(index, "hnsw"):
        index.hnsw.efSearch = EF_SEARCH

    if hasattr(index, "nprobe"):
        index.nprobe = IVF_NPROBE

    return index


def search_index_file(index_path, index_name, xq, topk=10):
    index_path = Path(index_path)

    t0 = time.perf_counter()
    index = faiss.read_index(str(index_path))
    load_sec = time.perf_counter() - t0

    index = set_search_params(index)

    print(index_name)
    print("path:", index_path)
    print("type:", type(index))
    print("dim:", index.d)
    print("ntotal:", index.ntotal)

    assert index.d == xq.shape[1], f"Sai dim: index.d={index.d}, xq={xq.shape[1]}"
    assert index.ntotal > 0, "Index rỗng"

    index.search(xq[:10], topk)

    t0 = time.perf_counter()
    D, I = index.search(xq, topk)
    search_sec = time.perf_counter() - t0

    row = {
        "index_name": index_name,
        "ntotal": index.ntotal,
        "dim": index.d,
        "file_size_mb": round(index_path.stat().st_size / 1024 / 1024, 2),
        "load_sec": round(load_sec, 4),
        "search_sec": round(search_sec, 4),
        "ms_per_query": round(search_sec * 1000 / len(xq), 4),
        "qps": round(len(xq) / search_sec, 2),
    }

    del index
    gc.collect()

    return row, D, I

In [31]:
def check_index_file(path, name):
    idx = faiss.read_index(str(path))

    print(name)
    print("path:", path)
    print("type:", type(idx))
    print("dim:", idx.d)
    print("ntotal:", idx.ntotal)
    print("valid:", idx.d == 384 and idx.ntotal > 0)
    print()

    d = idx.d
    ntotal = idx.ntotal

    del idx
    gc.collect()

    return d, ntotal


flat_d, flat_n = check_index_file(FLAT_PATH, "FLAT")
hnsw_d, hnsw_n = check_index_file(HNSW_PATH, "HNSW")

FLAT
path: /mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/flat_full.index
type: <class 'faiss.swigfaiss.IndexFlatIP'>
dim: 384
ntotal: 1347729
valid: True

HNSW
path: /mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/hnsw_full.index
type: <class 'faiss.swigfaiss.IndexHNSWFlat'>
dim: 384
ntotal: 1347729
valid: True



In [33]:
row_flat, D_flat, I_flat = search_index_file(
    str(FLAT_PATH),
    "FlatIP_full",
    xq,
    TOPK
)

row_flat

FlatIP_full
path: /mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/flat_full.index
type: <class 'faiss.swigfaiss.IndexFlatIP'>
dim: 384
ntotal: 1347729


{'index_name': 'FlatIP_full',
 'ntotal': 1347729,
 'dim': 384,
 'file_size_mb': 1974.21,
 'load_sec': 8.5362,
 'search_sec': 78.055,
 'ms_per_query': 156.11,
 'qps': 6.41}

In [36]:
row_hnsw, D_hnsw, I_hnsw = search_index_file(
    str(HNSW_PATH),
    "HNSW_full_ef64",
    xq,
    TOPK
)

row_hnsw

HNSW_full_ef64
path: /mnt/c/Project/school/bigdata/BigData_Nhom_6/scripts/benchmark/recommendation/indexes/hnsw_full.index
type: <class 'faiss.swigfaiss.IndexHNSWFlat'>
dim: 384
ntotal: 1347729


{'index_name': 'HNSW_full_ef64',
 'ntotal': 1347729,
 'dim': 384,
 'file_size_mb': 2324.01,
 'load_sec': 4.3758,
 'search_sec': 0.1113,
 'ms_per_query': 0.2226,
 'qps': 4491.57}